**Tibor3m swap using Tona curve**

In [1]:
tnDATA = [('depo','1d',0.7280), ('swap','1w',0.7277), ('swap','2w',0.72875),
          ('swap','3w',0.72875),('swap','1m',0.72875),('swap','2m',0.72956),
          ('swap','3m',0.7414), ('swap','4m',0.76665),('swap','5m',0.79104),
          ('swap','6m',0.81747),('swap','7m',0.84223),('swap','8m',0.86397),
          ('swap','9m',0.8872), ('swap','10m',0.91146),('swap','11m',0.9338),
          ('swap','12m',0.9575),('swap','15m',1.02),  ('swap','18m',1.0825),
          ('swap','2y',1.2025), ('swap','3y',1.3775), ('swap','4y',1.51375),
          ('swap','5y',1.6275), ('swap','6y',1.73125),('swap','7y',1.83125),]

tbDATA=[('depo','3m',1.07106),('fra','1m',1.1281),  ('fra','2m',1.1847),
        ('fra','3m',1.24155), ('fra','4m',1.285),   ('fra','5m',1.35253),
        ('fra','6m',1.42688), ('swap','1y',1.32625),('swap','18m',1.465),
        ('swap','2y',1.59875),('swap','3y',1.79875),('swap','4y',1.9525),
        ('swap','5y',2.09),   ('swap','6y',2.20625),('swap','7y',2.31875),]

In [ ]:
from myABBR import * ; import myUTIL as mu
trdDT = jDT(2026,1,20) ; setEvDT(trdDT)
tnIX,tnCvOBJ,tnCvHDL,tnParRT = mu.makeTonaCurve (tnDATA)
tbIX,tbCvOBJ,tbCvHDL,tbParRT = mu.makeTiborCurve(tbDATA, dCRV=tnCvHDL, iTNR=3)
# check
dfTN = mu.dfNodes(tnCvOBJ,tnParRT,cmpd=CNT);print('(Tona)');   dfDSP(dfTN,5)
dfTB = mu.dfNodes(tbCvOBJ,tbParRT,cmpd=CNT);print('(3mTibor)');dfDSP(dfTB,5)
mu.showCurveDate(tbIX, tbCvOBJ)

(Tona)


,date,matYR,parRT,zeroRT,DF
0,2026-01-20,0.0000,nan%,0.727993%,1.00000000
1,2026-01-21,0.0027,0.728000%,0.727993%,0.99998006
2,2026-01-29,0.0247,0.727700%,0.727687%,0.99982059
3,2026-02-05,0.0438,0.728750%,0.728545%,0.99968069
4,2026-02-12,0.0630,0.728750%,0.728530%,0.99954103
20,2029-01-22,3.0082,1.377500%,1.370440%,0.95961239
21,2030-01-22,4.0082,1.513750%,1.507337%,0.94137154
22,2031-01-22,5.0082,1.627500%,1.622316%,0.92196397
23,2032-01-22,6.0082,1.731250%,1.727983%,0.90138661
24,2033-01-24,7.0164,1.831250%,1.830735%,0.87945543


(3mTibor)


,date,matYR,parRT,zeroRT,DF
0,2026-01-22,0.0000,nan%,1.069648%,1.00000000
1,2026-04-22,0.2466,1.071060%,1.069648%,0.99736599
2,2026-05-25,0.3370,1.128100%,1.111272%,0.99626217
3,2026-06-23,0.4164,1.184700%,1.138216%,0.99527125
4,2026-07-22,0.4959,1.241550%,1.155110%,0.99428830
11,2029-01-22,3.0027,1.798750%,1.795933%,0.94750108
12,2030-01-22,4.0027,1.952500%,1.951138%,0.92487287
13,2031-01-22,5.0027,2.090000%,2.090980%,0.90067906
14,2032-01-22,6.0027,2.206250%,2.210023%,0.87576109
15,2033-01-24,7.0110,2.318750%,2.326360%,0.84950611


tradeDT:2026-01-20, settleDT:2026-01-22, index:Tibor3M Actual/365 (Fixed)


* 2カーブ Tibor3m swap

In [3]:
effDT,                matDT,      ntlAMT,     cpnRT,                      = \
jDT(2026,1,22), jDT(2031,1,22), 10_000_000,  2.09/100

# スケジュール, swapオブジェクト、エンジン
fixSCD  = ql.Schedule(effDT, matDT, pDfrqSA, calJP, mFLLW, mFLLW, dtGENb, EoMf)
fltSCD  = ql.Schedule(effDT, matDT, pDfrqQ,  calJP, mFLLW, mFLLW, dtGENb, EoMf)
tbSwOBJ = ql.VanillaSwap(swPAY, ntlAMT, fixSCD, cpnRT,        dcA365,
                                        fltSCD, tbIX, spdRT0, dcA365)
tbSwENG = ql.DiscountingSwapEngine(tnCvHDL); tbSwOBJ.setPricingEngine(tbSwENG)

print(f'固定レグ:{tbSwOBJ.legNPV(0):,.3f}, ' f'変動レグ:{tbSwOBJ.legNPV(1):,.3f}, '
      f'スワップNPV:{tbSwOBJ.NPV():,.3f}, '  f'フェアレート:{tbSwOBJ.fairRate():.6%}')

固定レグ:-1,005,794.097, 変動レグ:1,005,794.097, スワップNPV:-0.000, フェアレート:2.090000%


In [4]:
dfL1 = mu.swapCashFlow(tbSwOBJ,tbCvOBJ,leg=1); dfDSP(dfL1,5)

,fixDate,accStart,accEnd,payDate,days,rate,spread,amount,DF
0,nan,nan,nan,2026-01-22,nan,nan%,nan%,nan,1.00000000
1,2026-01-20,2026-01-22,2026-04-22,2026-04-22,90,1.071060%,0.000%,"26,409.70",0.99736599
2,2026-04-20,2026-04-22,2026-07-22,2026-07-22,91,1.241550%,0.000%,"30,953.71",0.99428830
3,2026-07-17,2026-07-22,2026-10-22,2026-10-22,92,1.426880%,0.000%,"35,965.19",0.99072513
4,2026-10-20,2026-10-22,2027-01-22,2027-01-22,92,1.554368%,0.000%,"39,178.60",0.98685876
16,2029-10-18,2029-10-22,2030-01-22,2030-01-22,92,2.424556%,0.000%,"61,112.11",0.92487287
17,2030-01-18,2030-01-22,2030-04-22,2030-04-22,90,2.659412%,0.000%,"65,574.54",0.91884757
18,2030-04-18,2030-04-22,2030-07-22,2030-07-22,91,2.659509%,0.000%,"66,305.56",0.91279523
19,2030-07-18,2030-07-22,2030-10-22,2030-10-22,92,2.659605%,0.000%,"67,036.62",0.90671691
20,2030-10-18,2030-10-22,2031-01-22,2031-01-22,92,2.659605%,0.000%,"67,036.62",0.90067906


In [5]:
# (hc) fwd rates
nA( (dfL1.DF.shift(1)/dfL1.DF-1)/(dfL1.days/365)*100 )

array([    nan, 1.07106, 1.24155, 1.42688, 1.55437, 1.74439, 1.74443,
       1.99297, 1.99308, 2.20573, 2.20573, 2.20573, 2.20573, 2.42448,
       2.42448, 2.42448, 2.42456, 2.65941, 2.65951, 2.65961, 2.65961])

* 下のセルのように、TiborのDFからスワップレートは計算出来ない

In [6]:
# (hc) 5y Tibor swapRT
tbAnn = np.sum(dfL1.DF[1:]*dfL1.days[1:]/365)
tbSw5y = (1-dfL1.DF.iloc[-1])/tbAnn
print(f'TiborAnnuity:{tbAnn:.5f}, ' f'5ySwapRT:{tbSw5y:.5%}')

TiborAnnuity:4.76904, 5ySwapRT:2.08262%
